In [ ]:
import pandas as pd
import numpy as np
import re
import ast
from html import unescape
import unicodedata
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

In [ ]:
df_prod_geo_macro = pd.read_csv ('external_data/df_prod_geo_macro.csv', low_memory=False)

In [ ]:
df_prod_geo_macro.info()

# Подготовка данных для этапа M3 Text.
Витрина df_prod_geo_macro_text

In [ ]:
df_prod_geo_macro_text = df_prod_geo_macro.copy()
df_prod_geo_macro_text = df_prod_geo_macro_text.reset_index(drop=True)
df_prod_geo_macro_text.insert(0, "row_id", np.arange(len(df_prod_geo_macro_text), dtype=np.int64))
print(df_prod_geo_macro_text.shape)

## Базовый контроль качества

In [ ]:
# --- Наличие обязательных колонок ---
required_cols = [
    "row_id",
    "name",
    "description",
    "key_skills",
    "salary_from_log",
    "region_name",
]
missing = [c for c in required_cols if c not in df_prod_geo_macro_text.columns]

In [ ]:
check_na = ["name", "description", "salary_from_log", "region_name"]
na_counts = df_prod_geo_macro_text[check_na].isna().sum()
print("Пропуски по обязательным полям (без key_skills):")
print(na_counts)
assert na_counts.sum() == 0, "Есть пропуски в name / description / salary_from_log / region_name"
# --- Сводка ---
print(f"\nСтрок: {len(df_prod_geo_macro_text):,}")
print(f"Уникальных region_name: {df_prod_geo_macro_text['region_name'].nunique():,}")


## Предварительный анализ сырого текста

In [ ]:
# --- Доля пустых / отсутствующих навыков ---
ks = df_prod_geo_macro_text["key_skills"]
empty_skills = ks.isna() | (ks.astype(str).str.strip() == "")
print(
    f"Доля вакансий без навыков (NaN или пусто): {empty_skills.mean():.4%} "
    f"({empty_skills.sum():,} / {len(df_prod_geo_macro_text):,})"
)
# --- Длины в символах ---
def describe_lengths(series: pd.Series, label: str, only_non_null: bool = False):
    if only_non_null:
        s = series.dropna().astype(str)
    else:
        s = series.astype(str)
    lens = s.str.len()
    print(f"\n=== Длина в символах: {label} ===")
    print(lens.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
describe_lengths(df_prod_geo_macro_text["name"], "name", only_non_null=False)
describe_lengths(df_prod_geo_macro_text["description"], "description", only_non_null=False)
describe_lengths(df_prod_geo_macro_text["key_skills"], "key_skills (non-null)", only_non_null=True)

In [ ]:
# проанализируем все уникальные печатные символы
PRINTABLE_CHARS = (
    "".join(chr(i) for i in range(32, 127))      # ASCII печатные
    + "".join(chr(i) for i in range(128, 10000))  # как в вашем примере (расширение)
)
PRINTABLE_SET = set(PRINTABLE_CHARS)
def printable_char_alphabet(series: pd.Series, label: str):
    """Уникальные символы в серии: список «печатных» и символы вне PRINTABLE_SET."""
    s = series.dropna().astype(str)
    all_chars = set()
    for text in s:
        all_chars.update(text)
    printable_chars = sorted(c for c in all_chars if c in PRINTABLE_SET)
    non_printable = sorted((c for c in all_chars if c not in PRINTABLE_SET), key=ord)
    print(f"Всего уникальных символов: {len(all_chars)}")
    print(f"Считаются печатными (в PRINTABLE_CHARS): {len(printable_chars)}")
    print("Список печатных символов (отсортировано):")
    print(printable_chars)
    if non_printable:
        print(f"\nСимволы ВНЕ заданного набора PRINTABLE_CHARS: {len(non_printable)}")
        for c in non_printable[:500]:
            print(repr(c), "U+%04X" % ord(c), unicodedata.category(c), unicodedata.name(c, "?"))
        if len(non_printable) > 500:
            print(f"... и ещё {len(non_printable) - 80}")
printable_char_alphabet(df_prod_geo_macro_text["name"], "name")
printable_char_alphabet(df_prod_geo_macro_text["description"], "description")
printable_char_alphabet(df_prod_geo_macro_text["key_skills"], "key_skills")

In [ ]:
# Символы, которые часто портят токенизацию, но почти не несут смысла
_ZERO_WIDTH_AND_FORMAT = re.compile(
    "[\u200b\u200c\u200d\u200e\u200f\u2060\u2061\u2062\u2063\u2064\ufeff\u00ad]"
)

# «Пробелоподобные» из Unicode -> обычный пробел
_SPACE_LIKE = re.compile(
    "[\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]"
)


def _strip_noise_chars(s: str) -> str:
    """
    Убираем: PUA (Co), вариационные селекторы, U+FFFD, основные CJK/кана/хангыль,
    декоративные блоки BMP (стрелки, звёзды, галочки), флаги/эмодзи в доп. плоскости.
    Не претендует на полное удаление всех So в Юникоде — только типичный мусор в вакансиях.
    """
    out = []
    for c in s:
        cp = ord(c)
        cat = unicodedata.category(c)

        if cat == "Co":  # Private Use Area — почти всегда артефакт
            continue
        if cp in (0xFE0E, 0xFE0F):  # variation selectors (эмодзи/совместимость)
            continue
        if cp == 0xFFFD:  # replacement character
            continue

        # CJK Unified Ideographs + Extension A
        if 0x3400 <= cp <= 0x4DBF or 0x4E00 <= cp <= 0x9FFF:
            continue
        # Хирагана, катакана
        if 0x3040 <= cp <= 0x30FF:
            continue
        # Хангыль
        if 0xAC00 <= cp <= 0xD7AF:
            continue

        # Декоративные символы BMP: Misc symbols, Dingbats, стрелки и т.п.
        if 0x2600 <= cp <= 0x26FF or 0x2700 <= cp <= 0x27BF or 0x2B00 <= cp <= 0x2BFF:
            continue

        # Региональные индикаторы (пары → флаги в тексте)
        if 0x1F1E6 <= cp <= 0x1F1FF:
            continue
        # Основной массив эмодзи / пиктограмм (дополнительная плоскость)
        if 0x1F300 <= cp <= 0x1FAFF:
            continue

        out.append(c)
    return "".join(out)


def clean_text_rubert(s: str) -> str:
    """
    Подготовка текста для ruBERT *cased*: без .lower() и без лемматизации.
    NFC → HTML → невидимки/пробелы → склейки/теги → шумовые символы → NFKC → пробелы.
    """
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFC", s)
    s = unescape(s)
    s = _ZERO_WIDTH_AND_FORMAT.sub("", s)
    s = _SPACE_LIKE.sub(" ", s)

    s = re.sub(r"([а-яё])([А-ЯЁ])", r"\1 \2", s)
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", s)
    s = re.sub(r"([а-яёa-zА-ЯЁA-Z])([()])", r"\1 \2", s)
    s = re.sub(r"([()])([а-яёa-zА-ЯЁA-Z])", r"\1 \2", s)

    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"[\n\t\r•*]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    s = _strip_noise_chars(s)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s

def clean_text_rubert_soft(s: str) -> str:
    """Та же подготовка, что до шагов _strip_noise_chars и NFKC (запасной путь для name)."""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFC", s)
    s = unescape(s)
    s = _ZERO_WIDTH_AND_FORMAT.sub("", s)
    s = _SPACE_LIKE.sub(" ", s)
    s = re.sub(r"([а-яё])([А-ЯЁ])", r"\1 \2", s)
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", s)
    s = re.sub(r"([а-яёa-zА-ЯЁA-Z])([()])", r"\1 \2", s)
    s = re.sub(r"([()])([а-яёa-zА-ЯЁA-Z])", r"\1 \2", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"[\n\t\r•*]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_text_rubert_name(s: str) -> str:
    """Жёсткая очистка; если заголовок обнулился (напр. только CJK), подставляем мягкую."""
    hard = clean_text_rubert(s)
    return hard if hard else clean_text_rubert_soft(s)


def key_skills_to_plain(val) -> str:
    """key_skills HH: строка или список dict с ключом 'name'; NaN / пусто -> ''."""
    if pd.isna(val):
        return ""
    if not isinstance(val, str):
        return clean_text_rubert(str(val))

    raw = val.strip()
    if not raw:
        return ""

    if raw.startswith("["):
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                parts = []
                for item in parsed:
                    if isinstance(item, dict) and "name" in item:
                        parts.append(str(item["name"]))
                    elif isinstance(item, str):
                        parts.append(item)
                joined = ", ".join(p.strip() for p in parts if p)
                return clean_text_rubert(joined)
        except (ValueError, SyntaxError):
            pass

    return clean_text_rubert(raw)


df_prod_geo_macro_text["name_clean"] = df_prod_geo_macro_text["name"].map(clean_text_rubert_name)
df_prod_geo_macro_text["description_clean"] = df_prod_geo_macro_text["description"].map(
    clean_text_rubert
)
df_prod_geo_macro_text["key_skills_clean"] = df_prod_geo_macro_text["key_skills"].map(
    key_skills_to_plain
)
df_prod_geo_macro_text["has_key_skills"] = (
    df_prod_geo_macro_text["key_skills_clean"].str.len() > 0
)


In [ ]:
assert (df_prod_geo_macro_text["name_clean"].str.len() > 0).all(), "Есть пустые name_clean"
assert (df_prod_geo_macro_text["description_clean"].str.len() > 0).all(), (
    "Есть пустые description_clean"
)

In [ ]:
df_prod_geo_macro_text[["name_clean", "key_skills_clean", "description_clean"]].head(5)

In [ ]:
# сделаем краткий проход по сборке text_for_model, используем токенизатор ruBERT
# создадим text_for_model как склейку name_clean / key_skills_clean / description_clean через sep_token

MODEL_NAME = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
SEP = tokenizer.sep_token

if SEP is None or not str(SEP).strip():
    raise ValueError("У токенизатора нет sep_token — проверьте MODEL_NAME и конфиг.")

def build_text_for_model(row: pd.Series) -> str:
    """name_clean → key_skills_clean → description_clean; только непустые части; join через sep_token."""
    parts = [
        row["name_clean"],
        row["key_skills_clean"],
        row["description_clean"],
    ]
    parts = [p for p in parts if isinstance(p, str) and p.strip()]
    return SEP.join(parts)
df_prod_geo_macro_text["text_for_model"] = df_prod_geo_macro_text.apply(
    build_text_for_model, axis=1
)
assert (df_prod_geo_macro_text["text_for_model"].str.len() > 0).all(), (
    "Есть пустой text_for_model"
)
print(
    df_prod_geo_macro_text[
        ["name_clean", "key_skills_clean", "description_clean", "text_for_model"]
    ].head(2)
)


Проанализируем распредление длин текста

In [ ]:
N_TOKEN_SAMPLE = 10_000  # None = все строки (долго на ~130k)
RANDOM_STATE = 42
MAX_LENGTH_REF = 512  # ориентир для rubert-base
s = df_prod_geo_macro_text["text_for_model"].astype(str)
char_len = s.str.len()

print("=== Длина в символах: text_for_model ===")
print(char_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(char_len, bins=80, color="steelblue", edgecolor="white", alpha=0.9)
axes[0].set_title("Распределение длины (символы)")
axes[0].set_xlabel("Символов")
axes[0].set_ylabel("Число вакансий")
bin_edges = [0, 100, 200, 300, 500, 800, 1200, 2000, char_len.max() + 1]
bin_edges = sorted(set(int(x) for x in bin_edges if x <= char_len.max() + 1))
if bin_edges[-1] <= char_len.max():
    bin_edges.append(int(char_len.max()) + 1)
labels = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    labels.append(f"{lo}–{hi-1}" if hi - lo > 1 else f"{lo}")
counts, _ = np.histogram(char_len, bins=bin_edges)
axes[1].bar(range(len(counts)), counts, color="coral", edgecolor="white")
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=45, ha="right")
axes[1].set_title("Длина в символах (диапазоны)")
axes[1].set_ylabel("Число вакансий")
plt.tight_layout()
plt.show()

In [ ]:
idx_full = df_prod_geo_macro_text.index
if N_TOKEN_SAMPLE is not None and N_TOKEN_SAMPLE < len(idx_full):
    idx = (
        df_prod_geo_macro_text["text_for_model"]
        .sample(n=N_TOKEN_SAMPLE, random_state=RANDOM_STATE)
        .index
    )
    print(f"\nТокены: сэмпл n={len(idx):,} из {len(idx_full):,}")
else:
    idx = idx_full
    print(f"\nТокены: полная выборка, n={len(idx):,}")
texts = df_prod_geo_macro_text.loc[idx, "text_for_model"].astype(str).tolist()
enc = tokenizer(
    texts,
    add_special_tokens=True,
    truncation=False,
    padding=False,
)
token_len = pd.Series(
    [len(ids) for ids in enc["input_ids"]],
    index=idx,
    name="token_len",
)
print("\n=== Длина в токенах: text_for_model (с CLS/SEP, без truncation) ===")
print(token_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
frac_over = (token_len > MAX_LENGTH_REF).mean()
print(
    f"\nДоля строк с длиной > {MAX_LENGTH_REF} токенов: {frac_over:.4%}"
)


~12% примеров усечены по 512 токенам; теряется конец текста

In [ ]:
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
axes2[0].hist(token_len, bins=80, color="seagreen", edgecolor="white", alpha=0.9)
axes2[0].set_title("Распределение длины (токены)")
axes2[0].set_xlabel("Токенов")
axes2[0].set_ylabel("Число вакансий (в выборке)")
axes2[0].axvline(MAX_LENGTH_REF, color="red", linestyle="--", linewidth=1, label=f"{MAX_LENGTH_REF}")
axes2[0].legend()
t_bins = [0, 128, 256, 384, 512, 768, 1024, int(token_len.max()) + 1]
t_bins = sorted(set(int(x) for x in t_bins))
if t_bins[-1] <= token_len.max():
    t_bins.append(int(token_len.max()) + 1)
t_labels = [f"{t_bins[i]}–{t_bins[i+1]-1}" for i in range(len(t_bins) - 1)]
t_counts, _ = np.histogram(token_len, bins=t_bins)
axes2[1].bar(range(len(t_counts)), t_counts, color="mediumpurple", edgecolor="white")
axes2[1].set_xticks(range(len(t_labels)))
axes2[1].set_xticklabels(t_labels, rotation=45, ha="right")
axes2[1].set_title("Длина в токенах (диапазоны)")
axes2[1].set_ylabel("Число вакансий (в выборке)")
plt.tight_layout()
plt.show()

In [ ]:
df_prod_geo_macro_text.to_csv('external_data/df_prod_geo_macro_text.csv', index=False)

## Создадим итоговый датасет для M3

In [ ]:
feature_text = [
    "region_name",
    "text_for_model"
]

target = "salary_from_log"

df_m3_text = df_prod_geo_macro_text[feature_text + [target]].copy()

In [ ]:
df_m3_text.to_excel('external_data/df_m3_text.xlsx', index=False)

### Контроль качества витрины M3 (df_m3_text)

In [ ]:
# --- QA-снимок df_m3_text до дедупликации (аналог df_m2_macro) ---
group_col = "region_name"
m3_snapshot_pre = {
    "n_rows": int(df_m3_text.shape[0]),
    "n_cols": int(df_m3_text.shape[1]),
    "n_regions": int(df_m3_text[group_col].nunique(dropna=True)),
    "target_name": target,
    "target_missing_pct": float(df_m3_text[target].isna().mean() * 100),
    "rows_with_any_na_pct": float(df_m3_text.isna().any(axis=1).mean() * 100),
    "text_for_model_empty_pct": float(
        (df_m3_text["text_for_model"].astype(str).str.strip().str.len() == 0).mean() * 100
    ),
}
display(pd.DataFrame([m3_snapshot_pre]))

In [ ]:
### Дедупликация и контроль витрины df_m3_text после дедупа
DEDUP_SUBSET = ["text_for_model", "salary_from_log", "region_name"]
n_before = len(df_m3_text)
n_dup = int(df_m3_text.duplicated(subset=DEDUP_SUBSET).sum())
print(f"Дубликаты по {DEDUP_SUBSET}: {n_dup:,}")
df_m3_text = df_m3_text.drop_duplicates(subset=DEDUP_SUBSET, keep="first").reset_index(drop=True)
n_after = len(df_m3_text)
print(f"После удаления дубликатов: {n_after:,} строк (убрано {n_before - n_after:,})")
post_dedup_report_m3 = pd.DataFrame(
    {
        "metric": [
            "n_rows_post_dedup",
            "n_regions_post_dedup",
            "duplicates_remaining",
            "target_missing_pct_post_dedup",
            "text_for_model_min_chars",
            "text_for_model_median_chars",
            "text_for_model_max_chars",
        ],
        "value": [
            int(df_m3_text.shape[0]),
            int(df_m3_text["region_name"].nunique(dropna=True)),
            int(df_m3_text.duplicated(subset=DEDUP_SUBSET).sum()),
            float(df_m3_text[target].isna().mean() * 100),
            int(df_m3_text["text_for_model"].str.len().min()),
            float(df_m3_text["text_for_model"].str.len().median()),
            int(df_m3_text["text_for_model"].str.len().max()),
        ],
    }
)
display(post_dedup_report_m3)
group_counts_m3 = (
    df_m3_text.groupby("region_name", dropna=False)
    .size()
    .reset_index(name="n_obs")
    .sort_values("n_obs", ascending=True)
)
display(group_counts_m3.head(15))
min_group_size = int(group_counts_m3["n_obs"].min())
n_groups = int(group_counts_m3["region_name"].nunique(dropna=False))
recommended_n_splits = max(2, min(5, min_group_size, n_groups))
print(f"Min group size: {min_group_size}")
print(f"Number of groups: {n_groups}")
print(f"Recommended n_splits for GroupKFold: {recommended_n_splits}")
if min_group_size < 5:
    print(
        "WARNING: Есть регионы с <5 наблюдениями. "
        "Для строгого GroupKFold(5) это методологический риск."
    )
assert df_m3_text["text_for_model"].str.len().gt(0).all(), "Пустой text_for_model"
assert df_m3_text[target].notna().all(), "Пропуски в salary_from_log"
assert df_m3_text["region_name"].notna().all(), "Пропуски в region_name"
assert df_m3_text["row_id"].is_unique, (
    "row_id не уникален после дедупа — проверьте критерий дедупа или ослабьте assert"
)
out_model = "data_for_models/df_m3_text.csv"
df_m3_text.to_csv(out_model, index=False)
print("Сохранено:", out_model, df_m3_text.shape)

In [ ]:
df_m3_text.to_csv("data_for_models/df_m3_text.csv", index=False)

In [ ]:
df_m3_text.info()